# CLO tranche modeling (two-factor intuition)

**Base correlation**, **subordination**, and **expected tranche loss** under alternative **two-factor** market scenarios using `TwoFactorModel.clo_standard()` and a one-factor Gaussian copula for defaults.

## Concept

**Base correlation** is a market quoting convention: implied correlation is backed out **per tranche** so that a pricing model reproduces observed tranche premiums. Economically, **senior** tranches are protected by **subordination**: they only take losses after **equity** and **mezzanine** absorb losses up to their **detachment** points.

A **two-factor** prepay–credit model (CLO standard calibration) describes correlated **prepayment** and **credit** shocks. For intuition, map independent normals $(Z_1, Z_2)$ to a **correlated credit systematic** draw using the published Cholesky loadings, then feed that into a **Gaussian copula** to obtain **conditional pool loss**. Conditional **tranche losses** follow the standard **waterfall** on **pool loss fraction** between **attachment** and **detachment**.

This notebook does **not** reprice a full CLO — it isolates **scenario** conditional **EL** patterns that explain why **equity** is convex to bad credit states while **super-senior** pieces are comparatively stable.

## API walkthrough

- **`TwoFactorModel.clo_standard()`** — CLO-oriented calibration (`prepay_vol=0.15`, `credit_vol=0.30`, factor correlation `-0.20`). Use **`cholesky_l10`** and **`cholesky_l11`** so the **credit** systematic is $Z_c = L_{10} Z_1 + L_{11} Z_2$ with $(Z_1,Z_2)$ independent standard normals.
- **`CopulaSpec.gaussian().build()`** — `conditional_default_prob(c, [Z_c], rho)` for a **homogeneous** pool with asset correlation `rho`.
- **`RecoverySpec`** — here a **constant** recovery spec builds a **`RecoveryModel`**; we use it only to fix **LGD** for pool loss fractions (optional pedagogical link to the recovery notebook).

In [1]:
from finstack.valuations.correlation import TwoFactorModel, CopulaSpec, RecoverySpec
from finstack.core.math.special_functions import standard_normal_inv_cdf

tf = TwoFactorModel.clo_standard()
print(f"CLO standard: {tf}")
print(
    f"Cholesky credit map: Z_c = {tf.cholesky_l10:.4f}*Z1 + {tf.cholesky_l11:.4f}*Z2"
)

gauss = CopulaSpec.gaussian().build()
rec = RecoverySpec.constant(0.40).build()
print(f"Copula: {gauss.model_name}; recovery model: {rec}")

CLO standard: TwoFactorModel(prepay=0.1500, credit=0.3000, corr=-0.2000)
Cholesky credit map: Z_c = -0.2000*Z1 + 0.9798*Z2
Copula: One-Factor Gaussian Copula; recovery model: RecoveryModel('Constant Recovery', expected=0.4000)


Helper: pool **loss fraction** $\ell \approx \mathrm{PD}(Z_c)\cdot \mathrm{LGD}$ (homogeneous large pool **EL** proxy). **Tranche** loss **fraction of tranche notional** with attach $A$ and detach $D$ (decimals of the pool):

$$L^{\mathrm{tr}} = \frac{\min(\max(\ell - A, 0),\, D-A)}{D-A}$$

In [2]:
def pool_loss_fraction(cond_pd: float, lgd: float) -> float:
    return min(max(cond_pd * lgd, 0.0), 1.0)


def tranche_loss_fraction(pool_ell: float, attach: float, detach: float) -> float:
    thick = detach - attach
    if thick <= 0.0:
        return 0.0
    penetr = min(max(pool_ell - attach, 0.0), thick)
    return penetr / thick


print(f"Example pool loss 8%: equity [0,3%] loss frac = {tranche_loss_fraction(0.08, 0.0, 0.03):.4f}")
print(f"Same pool: mezz [3,7%] loss frac = {tranche_loss_fraction(0.08, 0.03, 0.07):.4f}")
print(f"Same pool: senior [7,15%] loss frac = {tranche_loss_fraction(0.08, 0.07, 0.15):.4f}")

Example pool loss 8%: equity [0,3%] loss frac = 1.0000
Same pool: mezz [3,7%] loss frac = 1.0000
Same pool: senior [7,15%] loss frac = 0.1250


## Examples

Homogeneous pool: **200** names, **PD = 2.5%**, asset correlation **15%**, **LGD** from **40%** recovery. Tranches (pool **%** notionals): **equity 0–3%**, **mezz 3–7%**, **senior 7–15%**, **super-senior 15–100%**.

Scan **factor scenarios** $(Z_1, Z_2)$; print conditional **pool** loss and each **tranche** loss fraction.

In [3]:
n_pool = 200
pd_pool = 0.025
rho_asset = 0.15
lgd_pool = rec.lgd
c_pool = standard_normal_inv_cdf(pd_pool)

tranches = [
    ("equity", 0.0, 0.03),
    ("mezz", 0.03, 0.07),
    ("senior", 0.07, 0.15),
    ("super_senior", 0.15, 1.0),
]

# Stress primarily the credit factor Z2 (Zc = l10*Z1 + l11*Z2); benign is (0,0).
scenarios = [
    ("benign", 0.0, 0.0),
    ("mild", 0.0, -1.0),
    ("stress", 0.0, -2.0),
    ("severe", -2.0, -2.5),
]

print(
    f"Pool: n={n_pool}, PD={pd_pool}, rho={rho_asset}, LGD={lgd_pool:.2f} (constant recovery {rec.expected_recovery:.2f})"
)
print("Scenario | Z1    Z2    | Zc     | cond_PD | pool_loss | " + " | ".join(t[0] for t in tranches))

for label, z1, z2 in scenarios:
    zc = tf.cholesky_l10 * z1 + tf.cholesky_l11 * z2
    cond_pd = gauss.conditional_default_prob(c_pool, [zc], rho_asset)
    ell = pool_loss_fraction(cond_pd, lgd_pool)
    tr_parts = [f"{tranche_loss_fraction(ell, a, d):.4f}" for _, a, d in tranches]
    print(
        f"{label:12} {z1:5.2f} {z2:5.2f} | {zc:6.3f} | {cond_pd:7.4f} | {ell:9.4f} | "
        + " | ".join(tr_parts)
    )

z_bad = -3.0
zc_bad = tf.cholesky_l10 * z_bad + tf.cholesky_l11 * z_bad
ell_bad = pool_loss_fraction(
    gauss.conditional_default_prob(c_pool, [zc_bad], rho_asset), lgd_pool
)
print(
    f"Stress corner Z1=Z2={z_bad}: pool_loss={ell_bad:.4f}, "
    f"super_senior tranche loss frac={tranche_loss_fraction(ell_bad, 0.15, 1.0):.6f}"
)
print(
    "Seniority: equity absorbs first losses; super-senior remains near 0 until pool loss exceeds 15%."
)

Pool: n=200, PD=0.025, rho=0.15, LGD=0.60 (constant recovery 0.40)
Scenario | Z1    Z2    | Zc     | cond_PD | pool_loss | equity | mezz | senior | super_senior
benign        0.00  0.00 |  0.000 |  0.0168 |    0.0101 | 0.3351 | 0.0000 | 0.0000 | 0.0000
mild_stress  -1.00  0.50 |  0.690 |  0.0079 |    0.0047 | 0.1571 | 0.0000 | 0.0000 | 0.0000
credit_stress -2.00  1.00 |  1.380 |  0.0034 |    0.0020 | 0.0682 | 0.0000 | 0.0000 | 0.0000
severe       -2.50 -2.00 | -1.460 |  0.0652 |    0.0391 | 1.0000 | 0.2276 | 0.0000 | 0.0000
Stress corner Z1=Z2=-3.0: pool_loss=0.0759, super_senior tranche loss frac=0.000000
Seniority: equity absorbs first losses; super-senior remains near 0 until pool loss exceeds 15%.


## Takeaways

- **`TwoFactorModel.clo_standard()`** fixes a **prepay–credit** correlation structure; the **credit** systematic $Z_c$ is a **linear** combination of independent normals using **`cholesky_l10`** / **`cholesky_l11`**.
- **Subordination** allocates the same **pool** loss **sequentially**: **equity** is **first-loss**, **super-senior** is **last-loss** — scenario tables show **compression** of loss rates up the stack.
- **Base correlation** is a **per-tranche** implied parameter; a single **asset correlation** here is a **simplification** for exposition.
- Pair this with **`portfolio_default_simulation.ipynb`** for **simulated** loss distributions and with **`recovery_modeling.ipynb`** when **LGD** is **state-dependent**.